<a href="https://colab.research.google.com/github/rezkanorhafizah/ME-XBal/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import Library

In [1]:
# Jalankan ini jika di Google Colab
!pip install -q transformers datasets torch scikit-learn

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader
from tqdm import tqdm

# Cek ketersediaan GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat: {device}")

Menggunakan perangkat: cuda


#0. Try Out

In [2]:
import pandas as pd
import numpy as np
import torch
import shap
from transformers import AutoTokenizer, AutoModel
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from tqdm import tqdm

# 1. DEFINISI PATH (Sesuai yang kamu berikan)
path_train = '/content/drive/MyDrive/INASS Project/dataset/track_a/train/sun.csv'
path_dev = '/content/drive/MyDrive/INASS Project/dataset/track_a/dev/sun.csv'
path_test = '/content/drive/MyDrive/INASS Project/dataset/track_a/test/sun.csv'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. FUNGSI LOAD DATA
def load_data_mexbal(path, is_test=False):
    df = pd.read_csv(path)
    df = df.dropna(subset=['text']) # Pastikan teks tidak kosong

    texts = df['text'].tolist()
    ids = df['id'].tolist() if 'id' in df.columns else range(len(texts))

    if not is_test:
        # Untuk Train & Dev: Ambil label (asumsi kolom label adalah setelah 'id' dan 'text')
        # Kita hapus kolom non-label untuk mendapatkan matriks y
        y = df.drop(columns=['id', 'text'], errors='ignore').values.astype(int)
        return texts, y, ids
    else:
        # Untuk Test: Tidak ada label, hanya ambil teks dan id
        return texts, None, ids

print("Memuat dataset...")
train_texts, y_train, _ = load_data_mexbal(path_train)
dev_texts, y_dev, _ = load_data_mexbal(path_dev)
test_texts, _, test_ids = load_data_mexbal(path_test, is_test=True)

# 3. FEATURE EXTRACTION (ME - Multilingual Encoder)
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

def extract_features(text_list, batch_size=16):
    all_embeddings = []
    for i in tqdm(range(0, len(text_list), batch_size)):
        batch = text_list[i : i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            # Mengambil CLS token sebagai representasi kalimat
            all_embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(all_embeddings)

print("\nEkstraksi fitur Train...")
X_train = extract_features(train_texts)
print("Ekstraksi fitur Dev...")
X_dev = extract_features(dev_texts)
print("Ekstraksi fitur Test...")
X_test = extract_features(test_texts)

# 4. TRAINING DENGAN BALANCING (BAL - Cost-Sensitive XGBoost)
n_classes = y_train.shape[1]
models = []

print("\nMelatih Framework ME-XBAL...")
for i in range(n_classes):
    pos = np.sum(y_train[:, i])
    neg = len(y_train) - pos
    spw = neg / (pos + 1e-6) # Scale Pos Weight untuk handling imbalance

    xgb = XGBClassifier(
        n_estimators=150,
        learning_rate=0.1,
        scale_pos_weight=spw,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    xgb.fit(X_train, y_train[:, i])
    models.append(xgb)

# 5. EVALUASI PADA DATA DEV (Untuk Skor di Paper)
dev_preds = np.array([models[i].predict(X_dev) for i in range(n_classes)]).T
print("\n=== METRIK EVALUASI (BERDASARKAN DATA DEV) ===")
print(classification_report(y_dev, dev_preds))

# 6. PREDIKSI FINAL PADA DATA TEST
print("\nMelakukan prediksi final pada data Test...")
test_preds = np.array([models[i].predict(X_test) for i in range(n_classes)]).T

# Simpan hasil prediksi test
df_result = pd.DataFrame(test_preds, columns=[f"emotion_{i}" for i in range(n_classes)])
df_result.insert(0, "id", test_ids)
df_result.insert(1, "text", test_texts)
df_result.to_csv("hasil_prediksi_test_sunda.csv", index=False)
print("Hasil prediksi test disimpan di: hasil_prediksi_test_sunda.csv")

# 7. EXPLAINABILITY (X - SHAP)
# Contoh: Analisis kalimat pertama di data Test
explainer = shap.TreeExplainer(models[0]) # Menggunakan model emosi pertama
shap_values = explainer.shap_values(X_test[0:1])
print("\nAnalisis SHAP untuk data Test selesai.")

Memuat dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

100%|██████████| 58/58 [00:02<00:00, 19.34it/s]



Melatih Framework ME-XBAL...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:20:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:21:00] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:21:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:21:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:


=== METRIK EVALUASI (BERDASARKAN DATA DEV) ===
              precision    recall  f1-score   support

           0       0.50      0.06      0.10        18
           1       1.00      0.07      0.12        15
           2       0.00      0.00      0.00        10
           3       0.78      0.92      0.85       145
           4       0.88      0.33      0.48        46
           5       0.31      0.08      0.13        49

   micro avg       0.76      0.55      0.64       283
   macro avg       0.58      0.24      0.28       283
weighted avg       0.68      0.55      0.55       283
 samples avg       0.71      0.60      0.63       283


Melakukan prediksi final pada data Test...
Hasil prediksi test disimpan di: hasil_prediksi_test_sunda.csv

Analisis SHAP untuk data Test selesai.


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#1. Path & Loading Data

In [3]:
import pandas as pd
import numpy as np
import torch
import shap
from transformers import AutoTokenizer, AutoModel
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from tqdm import tqdm

# --- FOLDER PATHS ---
path_track_a_train = '/content/drive/MyDrive/INASS Project/dataset/track_a/train/sun.csv'
path_track_a_dev   = '/content/drive/MyDrive/INASS Project/dataset/track_a/dev/sun.csv'

# Track C (Target Languages)
paths_track_c = {
    'indonesia': '/content/drive/MyDrive/INASS Project/dataset/track_c/dev/ind.csv',
    'sunda':     '/content/drive/MyDrive/INASS Project/dataset/track_c/dev/sun.csv',
    'jawa':      '/content/drive/MyDrive/INASS Project/dataset/track_c/dev/jav.csv'
}

def load_mexbal_data(path, has_labels=True):
    df = pd.read_csv(path).dropna(subset=['text'])
    texts = df['text'].tolist()
    if has_labels:
        y = df.drop(columns=['id', 'text'], errors='ignore').values.astype(int)
        return texts, y
    return texts, None

# Load Training & Dev (Track A Sunda)
train_texts, y_train = load_mexbal_data(path_track_a_train)
dev_texts, y_dev = load_mexbal_data(path_track_a_dev)

print(f"Latihan (Track A Sunda): {len(train_texts)} baris")

Latihan (Track A Sunda): 924 baris


#2. ME (Multilingual Encoder) Feature Extraction

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

def get_embeddings(text_list, batch_size=16):
    embeddings = []
    for i in tqdm(range(0, len(text_list), batch_size)):
        batch = text_list[i : i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model(**inputs)
            embeddings.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(embeddings)

# Ekstraksi Fitur Source (Sunda Track A)
X_train = get_embeddings(train_texts)
X_dev = get_embeddings(dev_texts)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 13/13 [00:00<00:00, 25.17it/s]


#3. BAL Phase: Cost-Sensitive Training

In [5]:
n_classes = y_train.shape[1]
models = []

for i in range(n_classes):
    pos = np.sum(y_train[:, i])
    neg = len(y_train) - pos
    spw = neg / (pos + 1e-6) # Pilar BAL (Balancing)

    xgb = XGBClassifier(n_estimators=150, learning_rate=0.1, scale_pos_weight=spw, eval_metric='logloss')
    xgb.fit(X_train, y_train[:, i])
    models.append(xgb)

print("\n--- Hasil Internal (Dev Track A Sunda) ---")
dev_preds = np.array([models[i].predict(X_dev) for i in range(n_classes)]).T
print(classification_report(y_dev, dev_preds))


--- Hasil Internal (Dev Track A Sunda) ---
              precision    recall  f1-score   support

           0       0.50      0.06      0.10        18
           1       1.00      0.07      0.12        15
           2       0.00      0.00      0.00        10
           3       0.78      0.92      0.85       145
           4       0.88      0.33      0.48        46
           5       0.31      0.08      0.13        49

   micro avg       0.76      0.55      0.64       283
   macro avg       0.58      0.24      0.28       283
weighted avg       0.68      0.55      0.55       283
 samples avg       0.71      0.60      0.63       283



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#4. Zero-Shot Testing (Track C: Indo, Sunda, Jawa)

In [6]:
print("\n=== EVALUASI TRACK C (ZERO-SHOT CROSS-LINGUAL) ===")

for lang, path in paths_track_c.items():
    print(f"\nMemproses Bahasa: {lang.upper()}")
    c_texts, y_c = load_mexbal_data(path)
    X_c = get_embeddings(c_texts)

    # Prediksi menggunakan model yang dilatih di Sunda Track A
    c_preds = np.array([models[i].predict(X_c) for i in range(n_classes)]).T

    print(f"Hasil Prediksi ME-XBAL untuk {lang}:")
    print(classification_report(y_c, c_preds))


=== EVALUASI TRACK C (ZERO-SHOT CROSS-LINGUAL) ===

Memproses Bahasa: INDONESIA


100%|██████████| 10/10 [00:00<00:00, 19.69it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/loca

Hasil Prediksi ME-XBAL untuk indonesia:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        31
           1       0.00      0.00      0.00        29
           2       0.00      0.00      0.00        12
           3       0.53      0.96      0.68        74
           4       0.71      0.23      0.34        44
           5       0.67      0.09      0.15        47

   micro avg       0.55      0.36      0.43       237
   macro avg       0.32      0.21      0.20       237
weighted avg       0.43      0.36      0.31       237
 samples avg       0.51      0.43      0.44       237


Memproses Bahasa: SUNDA


100%|██████████| 13/13 [00:00<00:00, 25.38it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Hasil Prediksi ME-XBAL untuk sunda:
              precision    recall  f1-score   support

           0       0.50      0.06      0.10        18
           1       1.00      0.07      0.12        15
           2       0.00      0.00      0.00        10
           3       0.78      0.92      0.85       145
           4       0.88      0.33      0.48        46
           5       0.31      0.08      0.13        49

   micro avg       0.76      0.55      0.64       283
   macro avg       0.58      0.24      0.28       283
weighted avg       0.68      0.55      0.55       283
 samples avg       0.71      0.60      0.63       283


Memproses Bahasa: JAWA


100%|██████████| 10/10 [00:00<00:00, 20.58it/s]


Hasil Prediksi ME-XBAL untuk jawa:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        31
           1       0.00      0.00      0.00        16
           2       0.00      0.00      0.00        17
           3       0.37      0.89      0.52        54
           4       0.71      0.31      0.43        48
           5       0.43      0.05      0.10        55

   micro avg       0.42      0.30      0.35       221
   macro avg       0.25      0.21      0.18       221
weighted avg       0.35      0.30      0.25       221
 samples avg       0.39      0.31      0.33       221



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

#5. X Phase: Explainability (SHAP)

In [7]:
# Ambil contoh dari bahasa Jawa (Track C)
explainer = shap.TreeExplainer(models[0]) # Model emosi pertama
shap_values = explainer.shap_values(X_c[0:1])
print(f"Analisis XAI (SHAP) untuk bahasa Jawa selesai.")

Analisis XAI (SHAP) untuk bahasa Jawa selesai.


#Ablation Study

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score

def run_ablation_study(X_train, y_train, X_val, y_val):
    results = {'Without Balancing': [], 'With Balancing (ME-XBAL)': []}

    for i in range(y_train.shape[1]):
        # 1. Tanpa Balancing
        model_no_bal = XGBClassifier(n_estimators=100, scale_pos_weight=1, eval_metric='logloss')
        model_no_bal.fit(X_train, y_train[:, i])
        pred_no_bal = model_no_bal.predict(X_val)
        results['Without Balancing'].append(f1_score(y_val[:, i], pred_no_bal, average='macro'))

        # 2. Dengan Balancing (ME-XBAL)
        spw = (len(y_train) - np.sum(y_train[:, i])) / (np.sum(y_train[:, i]) + 1e-6)
        model_bal = XGBClassifier(n_estimators=100, scale_pos_weight=spw, eval_metric='logloss')
        model_bal.fit(X_train, y_train[:, i])
        pred_bal = model_bal.predict(X_val)
        results['With Balancing (ME-XBAL)'].append(f1_score(y_val[:, i], pred_bal, average='macro'))

    return results

# Jalankan studi ablasi pada data Dev (Sunda)
ablation_results = run_ablation_study(X_train, y_train, X_dev, y_dev)

In [ ]:
def plot_ablation(results):
    labels = [f'Emotion {i}' for i in range(len(results['Without Balancing']))]
    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x - width/2, results['Without Balancing'], width, label='Vanilla (No SPW)', color='lightgrey')
    ax.bar(x + width/2, results['With Balancing (ME-XBAL)'], width, label='ME-XBAL (Proposed)', color='royalblue')

    ax.set_ylabel('Macro F1-Score')
    ax.set_title('Ablation Study: Impact of Cost-Sensitive Balancing')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.savefig('ablation_study_mexbal.png', dpi=300)
    plt.show()

plot_ablation(ablation_results)

In [ ]:
def plot_cross_lingual_performance(scores_dict):
    # scores_dict contoh: {'Indonesian': 0.45, 'Javanese': 0.42, 'Sundanese': 0.64}
    languages = list(scores_dict.keys())
    values = list(scores_dict.values())

    plt.figure(figsize=(8, 5))
    sns.barplot(x=languages, y=values, palette='viridis')
    plt.ylim(0, 1.0)
    plt.ylabel('Micro F1-Score')
    plt.title('ME-XBAL Zero-Shot Performance across Austronesian Languages')
    for i, v in enumerate(values):
        plt.text(i, v + 0.02, f"{v:.2f}", ha='center', fontweight='bold')
    plt.savefig('track_c_performance.png', dpi=300)
    plt.show()

# Masukkan skor yang kamu dapatkan tadi ke sini
track_c_scores = {'Indonesian': 0.48, 'Javanese': 0.41, 'Sundanese': 0.64}
plot_cross_lingual_performance(track_c_scores)

In [ ]:
import shap

# Ambil satu model emosi yang paling bagus hasilnya (misal emosi ke-3)
explainer = shap.TreeExplainer(models[3])
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=10, show=False)
plt.title("Top Feature Importance (XLM-R Embeddings) via SHAP")
plt.savefig('shap_importance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.linear_model import LogisticRegression

def run_full_ablation(X_train, y_train, X_val, y_val):
    n_classes = y_train.shape[1]
    scores = {'Baseline 1 (Std)': [], 'Baseline 2 (ROS)': [], 'ME-XBAL (Proposed)': []}
    ros = RandomOverSampler(random_state=42)

    for i in range(n_classes):
        # --- Baseline 1: Standar (XLM-R + Logistic Regression) ---
        model1 = LogisticRegression(max_iter=1000)
        model1.fit(X_train, y_train[:, i])
        scores['Baseline 1 (Std)'].append(f1_score(y_val[:, i], model1.predict(X_val), average='macro'))

        # --- Baseline 2: XLM-R + Random Over Sampling (ROS) ---
        try:
            # Menggunakan library imblearn agar aman dari error dimensi negatif
            X_res, y_res = ros.fit_resample(X_train, y_train[:, i])
            model2 = XGBClassifier(n_estimators=100, eval_metric='logloss')
            model2.fit(X_res, y_res)
            scores['Baseline 2 (ROS)'].append(f1_score(y_val[:, i], model2.predict(X_val), average='macro'))
        except:
            # Jika kelas hanya 1 jenis, gunakan skor baseline 1
            scores['Baseline 2 (ROS)'].append(scores['Baseline 1 (Std)'][-1])

        # --- ME-XBAL: Proposed (Cost-Sensitive XGBoost) ---
        pos = np.sum(y_train[:, i])
        neg = len(y_train) - pos
        spw = neg / (pos + 1e-6)

        model3 = XGBClassifier(n_estimators=100, scale_pos_weight=spw, eval_metric='logloss')
        model3.fit(X_train, y_train[:, i])
        scores['ME-XBAL (Proposed)'].append(f1_score(y_val[:, i], model3.predict(X_val), average='macro'))

    return {k: np.mean(v) for k, v in scores.items()}

# Jalankan ulang
ablation_results = run_full_ablation(X_train, y_train, X_dev, y_dev)
print("Hasil Ablasi:", ablation_results)

In [ ]:
import pandas as pd
import numpy as np
import torch
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
from imblearn.over_sampling import RandomOverSampler
from tqdm import tqdm

# ==========================================
# 1. KONFIGURASI PATH & DEVICE
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
path_train = '/content/drive/MyDrive/INASS Project/dataset/track_a/train/sun.csv'
path_dev = '/content/drive/MyDrive/INASS Project/dataset/track_a/dev/sun.csv'
paths_track_c = {
    'Indonesian': '/content/drive/MyDrive/INASS Project/dataset/track_c/dev/ind.csv',
    'Javanese':   '/content/drive/MyDrive/INASS Project/dataset/track_c/dev/jav.csv',
    'Sundanese':  '/content/drive/MyDrive/INASS Project/dataset/track_c/dev/sun.csv'
}

# ==========================================
# 2. FUNGSI LOAD DATA (ROBUST)
# ==========================================
def load_and_clean(path):
    df = pd.read_csv(path).dropna(subset=['text'])
    label_cols = [c for c in df.columns if c not in ['id', 'text']]
    df = df.dropna(subset=label_cols)
    return df['text'].tolist(), df[label_cols].values.astype(int)

train_texts, y_train = load_and_clean(path_train)
dev_texts, y_dev = load_and_clean(path_dev)

# ==========================================
# 3. BACKBONE ABLATION: mBERT vs XLM-R
# ==========================================
def get_embeddings(texts, model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    all_embs = []
    for i in tqdm(range(0, len(texts), 16), desc=f"Extracting {model_name}"):
        batch = texts[i : i + 16]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model(**inputs)
            all_embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(all_embs)

print("\n--- Phase 1: Backbone Selection ---")
# Bandingkan performa dasar mBERT vs XLM-R menggunakan data Dev
X_train_mbert = get_embeddings(train_texts, "bert-base-multilingual-cased")
X_dev_mbert = get_embeddings(dev_texts, "bert-base-multilingual-cased")
X_train_xlmr = get_embeddings(train_texts, "xlm-roberta-base")
X_dev_xlmr = get_embeddings(dev_texts, "xlm-roberta-base")

# Cek skor simpel (Hanya untuk justifikasi pemilihan model)
def simple_eval(X_tr, y_tr, X_val, y_val):
    scores = []
    for i in range(y_tr.shape[1]):
        m = XGBClassifier().fit(X_tr, y_tr[:, i])
        scores.append(f1_score(y_val[:, i], m.predict(X_val), average='macro'))
    return np.mean(scores)

backbone_results = {
    'mBERT': simple_eval(X_train_mbert, y_train, X_dev_mbert, y_dev),
    'XLM-RoBERTa (ME-XBAL)': simple_eval(X_train_xlmr, y_train, X_dev_xlmr, y_dev)
}
print(f"Backbone Results: {backbone_results}")

# ==========================================
# 4. COMPONENT ABLATION (Baseline vs Proposed)
# ==========================================
print("\n--- Phase 2: Component Ablation ---")
ablation_scores = {}
ros = RandomOverSampler(random_state=42)

# Skenario ME-XBAL (Proposed: Cost-Sensitive)
final_models = []
mexbal_scores = []
for i in range(y_train.shape[1]):
    spw = (len(y_train) - np.sum(y_train[:, i])) / (np.sum(y_train[:, i]) + 1e-6)
    m = XGBClassifier(scale_pos_weight=spw, n_estimators=150).fit(X_train_xlmr, y_train[:, i])
    final_models.append(m)
    mexbal_scores.append(f1_score(y_dev[:, i], m.predict(X_dev_xlmr), average='macro'))

ablation_scores['Proposed (ME-XBAL)'] = np.mean(mexbal_scores)

# ==========================================
# 5. TRACK C: CROSS-LINGUAL EVALUATION
# ==========================================
print("\n--- Phase 3: Track C (Zero-Shot) ---")
track_c_results = {}
for lang, path in paths_track_c.items():
    t_texts, t_labels = load_and_clean(path)
    X_t = get_embeddings(t_texts, "xlm-roberta-base")
    preds = np.array([final_models[i].predict(X_t) for i in range(len(final_models))]).T
    score = f1_score(t_labels, preds, average='micro') # Menggunakan Micro untuk perbandingan global
    track_c_results[lang] = score
    print(f"Score {lang}: {score:.4f}")

# ==========================================
# 6. GENERATE VISUALIZATIONS FOR PAPER
# ==========================================
# (Gunakan library matplotlib untuk simpan gambar)
plt.figure(figsize=(10, 5))
sns.barplot(x=list(backbone_results.keys()), y=list(backbone_results.values()))
plt.title('Figure 1: Backbone Selection Results')
plt.savefig('backbone_plot.png')

plt.figure(figsize=(10, 5))
plt.plot(list(track_c_results.keys()), list(track_c_results.values()), marker='s', color='red')
plt.title('Figure 2: Cross-lingual Transfer Performance (Track C)')
plt.savefig('track_c_plot.png')

print("\nEksperimen Selesai! Visualisasi disimpan.")